# N03 - Watch the P0 pilot
Run on the ARCC2 login node (or through Open OnDemand) while the 78 jobs run. Read-only.

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get("HEIRLOOM_REPO", Path.cwd().parents[1]))
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import pandas as pd
AGG = REPO / "results" / "P0" / "aggregate"
print("repo:", REPO)

In [ ]:
import subprocess
print(subprocess.run(["squeue", "-u", os.environ.get("USER", "mahmudmi")], capture_output=True, text=True).stdout)

In [ ]:
# which per-job results exist so far
ps = REPO / "results/P0/per_seed"
done = sorted(p.stem for p in ps.glob("*.json")) if ps.exists() else []
print(f"{len(done)} job results written")
print(", ".join(done))

In [ ]:
# finished training runs: runtime and memory (Gate 1 criterion G1.6)
import json
rows = [json.loads(p.read_text()) for p in sorted((REPO / "results/P0/diagnostics").glob("train_*.json"))]
pd.DataFrame([{"branch": r["branch"], "seed": r["seed"], **{k: r["stats"][k] for k in
              ("optimizer_steps", "seconds_per_step", "peak_gpu_memory_gb")}} for r in rows])

In [ ]:
# any failed Slurm task?
import subprocess
print(subprocess.run("grep -l -iE 'error|traceback|CUDA out of memory' logs/slurm/*.out | tail -20",
                     shell=True, capture_output=True, text=True).stdout or "no failures found")